# Handling Missing Values, Data Cleaning, and Transformation

This notebook demonstrates how the `reduced_data` file was cleaned and transformed to create a consistent analysis-ready dataset.

In [2]:
import pandas as pd
from pathlib import Path

source = Path(r"C:\Users\amissetu\Documents\GitHub\final_project\wattwise-energy-analytics\reduced_data")
output = Path(r"C:\Users\amissetu\Documents\GitHub\final_project\wattwise-energy-analytics\cleaned_data_power_consumption")

# Load data
df = pd.read_csv(source)

# Confirm key columns exist
required = ['date', 'time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Treat missing or invalid values as null for all power metrics
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Create a single datetime column for consistency
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Remove rows with invalid datetime values
df = df.dropna(subset=['datetime']).copy()

# Remove duplicate timestamps
df = df.drop_duplicates(subset='datetime', keep='last').sort_values('datetime').reset_index(drop=True)

# Filter out rows where all the measurement columns are missing
df = df.dropna(subset=numeric_cols, how='all').reset_index(drop=True)

# Keep only the final analysis-ready columns
df = df[['datetime', *numeric_cols]].copy()
df.columns = [
    'datetime',
    'global_active_power',
    'global_reactive_power',
    'voltage',
    'global_intensity',
    'sub_metering_1',
    'sub_metering_2',
    'sub_metering_3'
]

# Save the cleaned data
df.to_csv(output, index=False, float_format='%.4f')

print('Rows after cleaning:', len(df))
print(df.head().to_string(index=False))


Rows after cleaning: 34168
           datetime  global_active_power  global_reactive_power  voltage  global_intensity  sub_metering_1  sub_metering_2  sub_metering_3
2006-12-16 17:00:00               4.2229                 0.2290 234.6439           18.1000             0.0          0.5278         16.8611
2006-12-16 18:00:00               3.6322                 0.0800 234.5802           15.6000             0.0          6.7167         16.8667
2006-12-16 19:00:00               3.4002                 0.0852 233.2325           14.5033             0.0          1.4333         16.6833
2006-12-16 20:00:00               3.2686                 0.0751 234.0715           13.9167             0.0          0.0000         16.7833
2006-12-16 21:00:00               3.0565                 0.0767 237.1587           13.0467             0.0          0.4167         17.2167
